# Reproducing Figure 4: Ar quasi-isotherms (Z vs. density)

This notebook is the acceptance test for `noblegasmd`: it calls `sweep()` to regenerate the
published quasi-isotherms of the compressibility factor `Z = PV/(NkT)` vs. number density for
argon at several temperatures, from the *J. Chem. Educ.* article this package ports
(Foley, Sweet, Akinfenwa). Run top-to-bottom in a fresh environment (including Google Colab)
with just `pip install noblegasmd`.

No LLM/API calls are used anywhere in the simulation loop -- `sweep()` is a thin Python
wrapper around a numba-jitted kernel.

In [1]:
# In Colab: !pip install noblegasmd
import numpy as np
import matplotlib.pyplot as plt

from noblegasmd import sweep

ModuleNotFoundError: No module named 'noblegasmd'

In [ ]:
# Full grid: takes a few minutes on a single Colab CPU (N=216, 20000 steps/run,
# 4 temperatures x 25 densities x 3 replicates = 300 runs).
# Reduce n_replicates or the density grid for a quick smoke test.
df = sweep(
    gas="Ar",
    T=[100, 200, 300, 400],
    rho=np.linspace(1.0, 5000.0, 25),
    n_replicates=3,
    seed=0,
)
df.head()

In [ ]:
# Ensemble mean +/- standard error across replicates, one isotherm per T.
summary = (
    df.groupby(["T_set", "rho_set"])["Z"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
summary["sem"] = summary["std"] / np.sqrt(summary["count"])

fig, ax = plt.subplots(figsize=(7, 5))
for T_set, group in summary.groupby("T_set"):
    ax.errorbar(
        group["rho_set"], group["mean"], yerr=group["sem"],
        marker="o", markersize=3, linewidth=1, capsize=2,
        label=f"T = {T_set:.0f} K",
    )
ax.axhline(1.0, color="gray", linestyle="--", linewidth=1, label="ideal gas (Z=1)")
ax.set_xlabel("number density (mol/m$^3$)")
ax.set_ylabel("Z = PV / (N k$_B$ T)")
ax.set_title("Argon quasi-isotherms (reproduction of Figure 4)")
ax.legend()
fig.tight_layout()
fig.savefig("figure4_reproduction.png", dpi=150)
plt.show()

## Acceptance check

Compares against the C oracle (`MD.cpp`) at a handful of state points, matching
`tests/test_vs_c_oracle.py`. This cell requires a C++ compiler and is optional --
skip it if you're only checking the qualitative isotherm shapes above (e.g. on Colab
without a working oracle build).

In [ ]:
import sys
from pathlib import Path

try:
    repo_root = Path.cwd().parent
    sys.path.insert(0, str(repo_root / "tests" / "oracle"))
    from oracle import run_oracle

    for T_set in [100.0, 200.0, 300.0, 400.0]:
        rho_set = 40.0
        python_z = df.loc[
            (df["T_set"] == T_set) & np.isclose(df["rho_set"], rho_set, atol=210), "Z"
        ]
        oracle_z = run_oracle("Ar", T_set, rho_set, title=f"nb_check_{T_set:.0f}").Z
        print(f"T={T_set:6.1f} K   oracle Z={oracle_z:.4f}   python nearby Z mean={python_z.mean():.4f}")
except (ImportError, FileNotFoundError, RuntimeError) as exc:
    print(f"Oracle comparison skipped ({exc}); the isotherm plot above is still valid.")